# Information from a table of counts

A support team logs every ticket with the channel it came in on, the customer's plan, the
weekday and the team that handled it, and whether the ticket was escalated. You will measure
how much each column tells you about escalation, using nothing but counts, and then how much a
column still tells you once another one is known.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 600 support tickets, every column an integer code.
#   channel    0 email, 1 chat, 2 phone
#   plan       0 free, 1 pro
#   weekday    0 Monday, ..., 6 Sunday
#   team       0 written support (email and chat), 1 phone support
#   escalated  the label: 0 no, 1 yes
rng = np.random.default_rng(7)
n = 600
channel = rng.choice(3, size=n, p=[0.5, 0.3, 0.2])
plan = (rng.random(n) < 0.4).astype(int)
weekday = rng.integers(0, 7, size=n)
team = (channel == 2).astype(int)
escalated = (rng.random(n) < np.array([0.08, 0.25, 0.55])[channel] + 0.10 * plan).astype(int)

X = np.column_stack([channel, plan, weekday, team])
names = ["channel", "plan", "weekday", "team"]
print(f"{n} tickets, {escalated.sum()} escalated")

## Part 1 (coding task): Mutual information from counts

Write `mutual_information(table)`. Its argument is a 2-D array of counts: `table[i, j]` is the
number of examples where the first variable takes value $i$ and the second takes value $j$.
Return $I$ between the two variables in **bits**.

The table holds counts, not probabilities, and it can contain zero cells and even whole rows of
zeros (a code that never occurs).

In [ ]:
def mutual_information(table):
    """I between the row variable and the column variable of a count table, in bits."""
    table = np.asarray(table, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
def close(a, b, tol=1e-9):
    return abs(float(a) - b) < tol

i_diag = mutual_information(np.array([[10, 0], [0, 30]]))
assert not np.isnan(i_diag), "a table with empty cells gave nan: an empty cell adds nothing (0 log 0 = 0)"
assert not close(i_diag, 0.5623351446188083), "that is the value in nats: use log base 2 for bits"
assert close(i_diag, 0.8112781244591328), f"got {i_diag}: when the table is diagonal, I is the entropy of either variable"
assert close(mutual_information(np.array([[6, 2], [3, 1]])), 0.0), "rows in the same proportion mean the variables are independent, so I = 0"
small = np.array([[2, 1], [1, 2]])
assert close(mutual_information(small), 0.08170416594551039), f"wrong I for {small.tolist()}: turn the counts into probabilities first"
assert close(mutual_information(7 * small), 0.08170416594551039), "multiplying every count by 7 should not change I"
assert close(mutual_information(np.array([[0, 0, 0], [4, 1, 3], [2, 5, 1]])), 0.20906112299665036), "wrong I when a whole row is empty"
assert close(mutual_information(np.array([[5, 1], [1, 5], [3, 3]])), 0.23331838556776374), "wrong I on a 3 x 2 table"
print("Part 1 passed")

## Part 2 (coding task): A table from two columns, and the gain of every feature

Write two functions.

- `contingency(a, b)` takes two integer arrays of equal length, with codes from 0 up, and
  returns the table of counts with `a.max() + 1` rows and `b.max() + 1` columns, as in Part 1.
- `information_gains(X, y)` returns an array with the information gain $I(X_{:, j}; y)$ of
  every column $j$ of `X`, in bits.

The check then prints the gain of each ticket column.

In [ ]:
def contingency(a, b):
    a, b = np.asarray(a), np.asarray(b)
    # TODO
    raise NotImplementedError


def information_gains(X, y):
    # TODO
    raise NotImplementedError

In [ ]:
t = contingency(np.array([0, 0, 1, 2, 2, 2]), np.array([1, 1, 0, 1, 1, 0]))
assert t.sum() == 6, f"the table counts {t.sum()} rows instead of 6: every repeated pair must be counted each time (a fancy-indexed += counts it once)"
assert np.array_equal(t, [[0, 2], [1, 0], [1, 2]]), f"wrong table {t.tolist()}: rows are the codes of a, columns the codes of b"
assert contingency(np.array([0, 3]), np.array([1, 1])).shape == (4, 2), "codes that never occur still need a row or a column"
Xs = np.array([[0, 1], [0, 0], [1, 1], [1, 0]])
g = np.asarray(information_gains(Xs, np.array([0, 0, 1, 1])))
assert g.shape == (2,), f"expected one gain per column, got shape {g.shape}"
assert np.allclose(g, [1.0, 0.0]), f"gains {g}: the first column decides y (1 bit), the second says nothing about it"

gains = information_gains(X, escalated)
for name, value in zip(names, gains):
    print(f"  {name:8s} {value:.4f} bits")
assert gains[0] == gains.max(), "channel should carry the most information about escalation"
print("Part 2 passed")

## Part 3 (coding task): What a column adds once another is known

The team column is built from the channel column. To see what a column adds beyond another,
use the conditional mutual information

$$
I(A; B \mid C) = \sum_c P(C = c)\, I(A; B \mid C = c),
$$

where $I(A; B \mid C = c)$ is the mutual information of $A$ and $B$ computed on the rows with
$C = c$ only. Write `conditional_mutual_information(a, b, c)` for three integer arrays of equal
length, in bits.

In [ ]:
def conditional_mutual_information(a, b, c):
    a, b, c = np.asarray(a), np.asarray(b), np.asarray(c)
    # TODO
    raise NotImplementedError

In [ ]:
a = np.array([0, 0, 1, 1]); c = np.array([0, 1, 0, 1]); b = a ^ c
assert close(mutual_information(contingency(a, b)), 0.0), "check Part 2 first: a and a XOR c share no information"
assert close(conditional_mutual_information(a, b, c), 1.0), "a and a XOR c are independent, but once c is known, b gives a away: expected 1 bit"
a2 = np.array([0, 1, 0, 0, 1, 1]); b2 = np.array([0, 1, 0, 1, 0, 1]); c2 = np.array([0, 0, 1, 1, 1, 1])
cmi = conditional_mutual_information(a2, b2, c2)
assert not close(cmi, 0.5), "a plain mean over the groups gives 0.5: weight each group by its share of the rows"
assert close(cmi, 1 / 3), f"got {cmi}, expected 1/3 (1 bit in a group with 2 of the 6 rows, 0 in the other)"

alone = mutual_information(contingency(team, escalated))
beyond = conditional_mutual_information(team, escalated, channel)
print(f"I(team; escalated)           = {alone:.4f} bits")
print(f"I(team; escalated | channel) = {beyond:.4f} bits")
assert close(beyond, 0.0, 1e-12), "team is a function of channel, so once channel is known it adds nothing"
print("Part 3 passed")

## Solution

### Part 1: Mutual information from counts

In [ ]:
def mutual_information(table):
    """I between the row variable and the column variable of a count table, in bits."""
    table = np.asarray(table, dtype=float)
    joint = table / table.sum()
    product = joint.sum(axis=1, keepdims=True) * joint.sum(axis=0, keepdims=True)
    nz = joint > 0                     # 0 log 0 = 0: empty cells add nothing
    return float(np.sum(joint[nz] * np.log2(joint[nz] / product[nz])))

**Why it works.** Dividing by the total turns counts into the joint distribution, and the row and
column sums are the two marginals. Their outer product, built here by broadcasting a column
against a row, is the joint the table would have if the variables were independent, and $I$ is
the KL from that product to the joint. Only cells with positive probability enter the sum, which
is the $0 \log 0 = 0$ convention, and a positive cell always has positive marginals, so nothing
divides by zero.

### Part 2: A table from two columns, and the gain of every feature

In [ ]:
def contingency(a, b):
    a, b = np.asarray(a), np.asarray(b)
    table = np.zeros((a.max() + 1, b.max() + 1), dtype=int)
    np.add.at(table, (a, b), 1)        # adds 1 once per row, repeated pairs included
    return table


def information_gains(X, y):
    X = np.asarray(X)
    return np.array([mutual_information(contingency(X[:, j], y)) for j in range(X.shape[1])])

**Why it works.** `np.add.at` adds 1 at `(a[k], b[k])` for every row $k$, so a pair that occurs
twice is counted twice. The shorter `table[a, b] += 1` does not do that: it reads the cells,
adds 1 and writes them back, so a repeated pair is written once. The gain of a column is then
Part 1 applied to its table against the label. On the tickets, channel leads with about 0.085
bits, team follows with 0.056 (it is part of channel), and weekday, which the data never uses,
still gets 0.004 bits: a plug-in estimate from a finite sample is almost never exactly zero.

### Part 3: What a column adds once another is known

In [ ]:
def conditional_mutual_information(a, b, c):
    a, b, c = np.asarray(a), np.asarray(b), np.asarray(c)
    total = 0.0
    for value in np.unique(c):
        rows = c == value
        total += rows.mean() * mutual_information(contingency(a[rows], b[rows]))
    return total

**Why it works.** Within the rows that share one value of $C$, Part 1 measures how much $A$
and $B$ still depend on each other, and the weights $P(C = c)$, the share of rows in each group,
average those numbers the way the definition says. Two cases from the check show why the
condition matters. In the XOR rows, $a$ alone says nothing about $b$, but inside each group of
$c$ it decides $b$ completely, so conditioning raised the information from 0 to 1 bit. The team
column goes the other way: it carries 0.056 bits alone, but within one channel it is constant,
so it adds nothing a tree that has already split on channel could use.